# Notebook 01 – Exploratory Data Analysis

**Dataset**: German Credit Data (UCI / StatLog) – 1 000 applicants, 20 features.

**Goal**: Understand class balance, distributions, correlations, and approval rates
across sensitive groups *before* touching any model.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

from src.data_prep import load_raw, engineer_features
from src.config import TARGET_COL, GENDER_COL, AGE_GROUP_COL, FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and inspect

In [ ]:
raw = load_raw()
df  = engineer_features(raw)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
df.dtypes

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values:\n', missing[missing > 0] if missing.sum() > 0 else 'None — dataset is complete.')

In [ ]:
df.describe(include='all').T.head(30)

**Takeaway**: The dataset has no missing values. Numerical features span wide ranges
(credit_amount 250–18 424 DM), suggesting we should scale them before modelling.
Categorical features use opaque codes (A11, A32, …) which are decoded in `config.py`.

## 2. Class balance

In [ ]:
counts = df[TARGET_COL].value_counts().rename({1: 'Approved', 0: 'Rejected'})
fig, ax = plt.subplots(figsize=(4, 3))
counts.plot(kind='bar', ax=ax, color=['#2ca02c', '#d62728'], edgecolor='black', width=0.5)
ax.set_title('Class balance')
ax.set_xlabel('')
ax.set_ylabel('Count')
ax.set_xticklabels(counts.index, rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())} ({p.get_height()/len(df):.0%})',
                (p.get_x() + p.get_width()/2, p.get_height() + 5), ha='center')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'class_balance.png', dpi=150)
plt.show()

**Takeaway**: 70% approved / 30% rejected — a moderate imbalance. We address this with
`class_weight='balanced'` in the models rather than oversampling, so the test set
reflects the real-world distribution.

## 3. Numerical feature distributions

In [ ]:
from src.config import NUM_FEATURES

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
axes = axes.flatten()
for i, col in enumerate(NUM_FEATURES):
    for label, grp in df.groupby(TARGET_COL):
        axes[i].hist(grp[col], bins=25, alpha=0.6,
                     label='Approved' if label == 1 else 'Rejected', density=True)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Numerical feature distributions by target', y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'num_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Takeaway**: Rejected applicants tend to have higher credit amounts and longer
durations. Age has a slight positive skew toward older applicants being approved.
`credit_amount` is heavily right-skewed — the 99th-percentile winsorisation in
`data_prep.clip_outliers` will reduce leverage from extreme values.

## 4. Categorical feature approval rates

In [ ]:
from src.config import CAT_FEATURES

# Show approval rate per category for the most important categorical features
top_cats = ['checking_status', 'credit_history', 'savings_status', 'employment_since']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()
for i, col in enumerate(top_cats):
    rates = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=True)
    rates.plot(kind='barh', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Approval rate by {col}', fontsize=10)
    axes[i].set_xlabel('Approval rate')
    axes[i].axvline(df[TARGET_COL].mean(), ls='--', color='red', lw=1, label='Overall avg')
    axes[i].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'cat_approval_rates.png', dpi=150, bbox_inches='tight')
plt.show()

**Takeaway**: `checking_status=A14` (no account) and `savings_status=A65` (no savings)
are not always associated with lower approval — complex interactions exist.
`credit_history=A34` (critical/other credits) shows a surprisingly *higher* approval
rate, possibly because these applicants request smaller amounts.

## 5. Correlation heatmap (numerical features)

In [ ]:
corr = df[NUM_FEATURES + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, ax=ax, linewidths=0.5)
ax.set_title('Pearson correlation – numerical features + target')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

**Takeaway**: `duration_months` and `credit_amount` are moderately correlated (r≈0.62),
which is expected — larger loans have longer terms. Neither is individually very
predictive of the target (|r| < 0.2), confirming we need the full feature set.

## 6. Approval rate by sensitive groups

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, col in zip(axes, [GENDER_COL, AGE_GROUP_COL, 'foreign_worker']):
    rates = df.groupby(col)[TARGET_COL].mean().sort_values()
    colors = plt.cm.Set2(np.linspace(0, 0.8, len(rates)))
    rates.plot(kind='bar', ax=ax, color=colors, edgecolor='black', width=0.5)
    ax.set_title(f'Approval rate by {col}', fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.axhline(df[TARGET_COL].mean(), ls='--', color='red', lw=1)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

fig.suptitle('Raw approval rates by sensitive group (pre-model)', y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_sensitive_groups.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for col in [GENDER_COL, AGE_GROUP_COL, 'foreign_worker']:
    print(f'\n{col}:')
    print(df.groupby(col)[TARGET_COL].agg(['mean', 'count']).rename(columns={'mean':'approval_rate', 'count':'n'}).round(3))

**Takeaway**: Males (≈75%) have a higher approval rate than females (≈67%) in the raw
data. The youngest group (<30) and foreign workers have slightly lower rates. These
differences may reflect genuine risk factors, socioeconomic inequality embedded in
historical data, or both — which is why a fairness audit on the *model* predictions
is essential (see Notebook 03).

## 7. Credit amount by gender

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df.boxplot(column='credit_amount', by=GENDER_COL, ax=ax, 
           boxprops=dict(color='steelblue'), medianprops=dict(color='red'))
ax.set_title('Credit amount by gender')
ax.set_xlabel('Gender')
ax.set_ylabel('Credit amount (DM)')
fig.suptitle('')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'credit_amount_by_gender.png', dpi=150)
plt.show()

**Takeaway**: Median credit amounts are similar across genders, but males show a
wider spread. This suggests that the approval-rate gap is not primarily driven by
loan size, pointing to other features (checking status, savings) as more likely
confounders.

## 8. EDA summary

| Finding | Action |
|---------|--------|
| 70/30 class imbalance | Use `class_weight='balanced'` in models |
| No missing values | No imputation needed, but pipeline still includes it for robustness |
| `credit_amount` right-skewed | Winsorise at 99th percentile |
| Gender gap in raw approval rate (~8 pp) | Fairness audit mandatory |
| High correlation: duration × amount | Keep both; tree models handle multicollinearity |
| Complex categorical patterns | Ordinal encoding + tree model preferred over one-hot + LR |